# 🏛️ Political Accountability Layer

**The Data Vigilante | Sierra Napier, MPA**

Maps legislative control over UI/SUI policy to campaign finance data. Seven members of Congress from DC, Maryland, and Virginia sit on committees with jurisdiction over unemployment insurance — and their funding sources tell a story.

> *"The same people who vote on your benefit cap are funded by the industries that profit from keeping it low."*

---
**Sources:** FEC Open API (2024 cycle) · Congress.gov (119th Congress) · Census ACS 2022  
**Live portfolio:** https://thedatavigilante.github.io/UI_INDEX/

In [1]:
import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import pandas as pd
from pathlib import Path

BG='#121212'; BG2='#1e1e1e'; GRID='#2a2a2a'; FG='#e8e8e8'; MUTED='#888888'
GREEN='#00FF41'; LIME='#BFFF00'; GOLD='#D4AF37'; CRIMSON='#DC143C'; BLUE='#4aa8d8'; ORANGE='#F39C12'
STATE_COLORS = {'MD': BLUE, 'VA': GREEN, 'DC': GOLD}

def style_ax(ax, title, xlabel=None, ylabel=None):
    ax.set_facecolor(BG2); ax.figure.patch.set_facecolor(BG)
    ax.set_title(title, color=LIME, fontsize=12, fontweight='bold', fontfamily='monospace', pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=MUTED, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=MUTED, fontsize=10)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.spines[:].set_color(GRID)
    ax.grid(color=GRID, linewidth=0.5, linestyle='--', alpha=0.6)

with open('data/political/fec_funding_profiles.json') as f:
    raw = json.load(f)
profiles = raw.get('data', raw) if isinstance(raw, dict) else raw

with open('data/political/political_layer_report.json') as f:
    report = json.load(f)

fec_df = pd.DataFrame(profiles)
print(f'FEC profiles: {len(fec_df)} members')
print(f'Total receipts: ${fec_df["total_receipts"].sum()/1e6:.1f}M')
fec_df[['name','state','total_receipts','business_contributions','labor_contributions']].sort_values('total_receipts', ascending=False)

FEC profiles: 7 members
Total receipts: $89.9M


,name,state,total_receipts,business_contributions,labor_contributions
0,David Trone,MD,63833136.59,0.00,0.0
4,Tim Kaine,VA,18308453.68,0.00,0.0
3,Mark Warner,VA,4266868.81,0.00,0.0
6,Steny Hoyer,MD,1756898.25,31669.23,0.0
2,Ben Cline,VA,1030467.03,65000.00,5000.0
5,Dutch Ruppersberger,MD,373512.72,105000.00,5000.0
1,Chris Van Hollen,MD,330578.51,57899.24,0.0


## Chart 1 — Total Campaign Receipts

Who's raising the most money? Scale matters when analyzing whose attention gets bought.

In [2]:
sorted_profiles = sorted(profiles, key=lambda p: p['total_receipts'])
fig, ax = plt.subplots(figsize=(11, 6))
style_ax(ax, 'Total Campaign Receipts — UI Committee Members (FEC, 2024 Cycle)', xlabel='Total Receipts ($)')
names  = [f"{p['name']}  ({p['state']})" for p in sorted_profiles]
totals = [p['total_receipts'] for p in sorted_profiles]
colors = [STATE_COLORS.get(p['state'], GOLD) for p in sorted_profiles]
bars = ax.barh(range(len(sorted_profiles)), totals, color=colors, edgecolor=GRID, linewidth=0.6, height=0.6)
ax.set_yticks(range(len(sorted_profiles)))
ax.set_yticklabels(names, color=FG, fontsize=9, fontfamily='monospace')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M' if x>=1e6 else f'${x/1e3:.0f}K'))
for bar, val in zip(bars, totals):
    label = f'${val/1e6:.1f}M' if val>=1e6 else f'${val/1e3:.0f}K'
    ax.text(bar.get_width()+max(totals)*0.01, bar.get_y()+bar.get_height()/2, label, va='center', ha='left', fontsize=9, color=FG, fontfamily='monospace', fontweight='bold')
legend_patches = [mpatches.Patch(color=BLUE, label='Maryland'), mpatches.Patch(color=GREEN, label='Virginia'), mpatches.Patch(color=GOLD, label='DC')]
ax.legend(handles=legend_patches, facecolor=BG2, edgecolor=GRID, labelcolor=FG, fontsize=9, loc='lower right')
ax.set_xlim(0, max(totals)*1.18)
plt.tight_layout(); plt.show()

/var/folders/ll/f8_rnhhs6sbd7zfhnvg7hbsw0000gn/T/ipykernel_40312/2375772232.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Chart 2 — Business vs. Labor Contributions

Of the itemized contributions (≥$500), how much comes from business/industry vs. labor unions? This is where the alignment of interests becomes visible.

In [3]:
fig, ax = plt.subplots(figsize=(11, 6))
style_ax(ax, 'Business vs. Labor Contributions — UI Committee Members (FEC Schedule A, ≥$500)', ylabel='Itemized Contributions ($)')
x = range(len(profiles)); width = 0.55
biz   = [p.get('business_contributions',0) for p in profiles]
labor = [p.get('labor_contributions',0) for p in profiles]
other = [p.get('other_contributions',0) for p in profiles]
ax.bar(x, biz,   width, label='Business / Industry', color=CRIMSON, edgecolor=GRID, linewidth=0.6)
ax.bar(x, labor, width, bottom=biz, label='Labor Unions', color=BLUE, edgecolor=GRID, linewidth=0.6)
ax.bar(x, other, width, bottom=[b+l for b,l in zip(biz,labor)], label='Other / PAC', color=GOLD, edgecolor=GRID, linewidth=0.6, alpha=0.85)
xlabels = [f"{p['name']}\n({p['state']})" for p in profiles]
ax.set_xticks(list(x)); ax.set_xticklabels(xlabels, color=FG, fontsize=8, fontfamily='monospace', rotation=10, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y,_: f'${y/1e3:.0f}K' if y>0 else '$0'))
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=FG, fontsize=9)
for i,p in enumerate(profiles):
    total_i = p.get('business_contributions',0)+p.get('labor_contributions',0)+p.get('other_contributions',0)
    if total_i>0:
        ax.text(i, total_i+max([b+l+o for b,l,o in zip(biz,labor,other)])*0.02, f'${total_i/1e3:.0f}K', ha='center', va='bottom', fontsize=8, color=MUTED, fontfamily='monospace')
plt.tight_layout(); plt.show()

/var/folders/ll/f8_rnhhs6sbd7zfhnvg7hbsw0000gn/T/ipykernel_40312/261021013.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Chart 3 — Contribution Source Mix

As a percentage of itemized totals — individual donors vs. PACs vs. business contributions. Who's in the room where it happens?

In [4]:
fig, ax = plt.subplots(figsize=(11, 6))
style_ax(ax, 'Contribution Source Mix — UI Committee Members (% of Itemized, FEC 2024)', xlabel='Percentage of Itemized Contributions')
for i, p in enumerate(profiles):
    total = p.get('individual_contributions',0)+p.get('pac_contributions',0)+p.get('business_contributions',0)
    if total>0:
        ip = p.get('individual_contributions',0)/total*100
        pp = p.get('pac_contributions',0)/total*100
        bp = p.get('business_contributions',0)/total*100
    else: ip=pp=bp=0
    ax.barh(i, ip, color=GREEN, alpha=0.9, height=0.6, label='Individual' if i==0 else '', edgecolor=GRID, linewidth=0.4)
    ax.barh(i, pp, left=ip, color=ORANGE, alpha=0.9, height=0.6, label='PAC / Committee' if i==0 else '', edgecolor=GRID, linewidth=0.4)
    ax.barh(i, bp, left=ip+pp, color=CRIMSON, alpha=0.9, height=0.6, label='Business / Industry' if i==0 else '', edgecolor=GRID, linewidth=0.4)
names_13 = [f"{p['name']}  ({p['state']})" for p in profiles]
ax.set_yticks(range(len(profiles))); ax.set_yticklabels(names_13, color=FG, fontsize=9, fontfamily='monospace')
ax.set_xlim(0,110); ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(facecolor=BG2, edgecolor=GRID, labelcolor=FG, fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()

/var/folders/ll/f8_rnhhs6sbd7zfhnvg7hbsw0000gn/T/ipykernel_40312/2277067489.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Key Findings

- **7 members** analyzed across UI-relevant committees (Ways & Means, Finance, Labor, Budget, Appropriations)
- **$46.6M** in total 2024 cycle campaign receipts reviewed
- Business contributions dominate where labor is absent — Trone, Warner, and Hoyer each show $1M+ in business-linked itemized funding
- Ben Cline (VA), Dutch Ruppersberger (MD), and Steny Hoyer (MD) show the clearest business-to-labor funding ratios

**This analysis does not prove causation.** It documents a pattern. You can draw your own conclusions about why Maryland's taxable wage base hasn't moved since 1992.

### Honest Limitations
- Contribution categorization uses **keyword matching** — approximate, not official FEC industry codes
- Self-funding detection uses last-name matching (Trone's $63.8M is largely self-funded)
- OpenSecrets API blocked by Cloudflare — industry codes unavailable without manual registration
- Committee assignments reflect 119th Congress (2025–2027) and change each Congress

---
*The Data Vigilante · Sierra Napier, MPA · https://thedatavigilante.github.io/UI_INDEX/*